In [1]:
pip install --upgrade torch ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.10.19
    Uninstalling nvidia-curand-cu12-10.3

In [2]:
from ultralytics import YOLO
import os, pandas, numpy, cv2
from pathlib import Path
from PIL import Image
import torch.nn as nn

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
CLASSES=['labels']

yaml_content = f"""
train: /kaggle/input/simulated-object-data/data/train/images
val: /kaggle/input/simulated-object-data/data/val/images

nc: {len(CLASSES)}
names: {CLASSES}
"""

with open("dataset.yaml", "w") as f:
    f.write(yaml_content)

print("dataset.yaml created!")

dataset.yaml created!


In [4]:
#model.add_module('custom_head', torch.nn.Sequential())

In [5]:
class Conv(nn.Module):
    default_act = nn.SiLU(inplace=True)

    def __init__(self, c1, c2, kernel_size=1, stride=1, padding=None, act=True, eps = 0.001, momentum = 0.03):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(c2, eps, momentum)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))
        
class RE_Conv(nn.Module):
    default_act = nn.SiLU(inplace=True)

    def __init__(self, c1, c2, kernel_size=1, stride=1, padding=None, act=True, eps = 0.001, momentum = 0.03):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(c2, eps, momentum)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.conv(x))

In [6]:
class C3k2(nn.Module):
    def __init__(self, x_0, x_1, x_2, x_3, x_4):
        super().__init__()

        self.cv1 = RE_Conv(x_0, x_1, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
        self.cv2 = RE_Conv(x_2, x_3, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))

        self.m = nn.ModuleList([
                      RE_Conv(x_4//2, x_4//4, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                      RE_Conv(x_4//2, x_4//4, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                      RE_Conv(x_4//2, x_4//2, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                
                      nn.Sequential(
                          RE_Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                          RE_Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                      
                          RE_Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                          RE_Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
                      )]
                  )
    def forward(self, x):
        x = self.cv1(x)
        x = self.cv2(x)
        return self.m(x)

In [7]:
yolo = YOLO("yolo11x.pt")

100%|██████████| 109M/109M [00:00<00:00, 226MB/s]


In [8]:
yolo.model.model[0] = Conv(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[1] = Conv(96, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[2] = C3k2(192, 192, 384, 384, 192)
yolo.model.model[3] = Conv(384, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[4] = C3k2(384, 384, 768, 768, 384)
yolo.model.model[5] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[6] = C3k2(768, 768, 1536, 768, 768)
yolo.model.model[7] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[8] = C3k2(768, 768, 1536, 768, 768)
yolo.model.model[13] = C3k2(1536, 768, 1536, 768, 768)
yolo.model.model[16] = C3k2(1536, 384, 768, 384, 384)
yolo.model.model[17] = Conv(384, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[19] = C3k2(1152, 768, 1152, 768, 768)
yolo.model.model[20] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[22] = C3k2(1536, 768, 1536, 768, 768)

In [9]:
yolo.train( data='/kaggle/working/dataset.yaml',
            epochs=150,
            batch=6,
            imgsz=724,
            patience=5,
            lr0=0.001,
            lrf=0.02,
            optimizer="SGD",
            momentum=0.96,
            weight_decay=0.001,
            cos_lr=True,
            dropout=0.3,
            label_smoothing=0.1,
            mosaic=0.5,
            mixup=0.15,
            copy_paste=0.1,
            fliplr=0.5,
            flipud=0.5,
            hsv_h=0.5,
            hsv_s=0.9,
            hsv_v=0.9,
            translate=0.2,
            scale=0.5,
            shear=0.2,
            perspective=0.0002,
            val=True,
            workers=8,
            seed=42,
            device=[-1, -1]
        )
valid_results = yolo.val()
print(valid_results)

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in in the future.
Searching for 2 idle GPUs with free memory >= 20.0% and free utilization >= 0.0%...
Selected idle CUDA devices [0, 1]
Ultralytics 8.3.159 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                        CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.5, hsv_s=0.9, hsv_v=0.9, imgsz=724, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf

100%|██████████| 755k/755k [00:00<00:00, 16.4MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      2784  ultralytics.nn.modules.conv.Conv             [3, 96, 3, 2]                 
  1                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  2                  -1  2    389760  ultralytics.nn.modules.block.C3k2            [192, 384, 2, True, 0.25]     
  3                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
  4                  -1  2   1553664  ultralytics.nn.modules.block.C3k2            [384, 768, 2, True, 0.25]     
  5                  -1  1   5309952  ultralytics.nn.modules.conv.Conv             [768, 768, 3, 2]              
  6                  -1  2   5022720  ultralytics.nn.modules.block.C3k2            [768, 768, 2, True]           
  7                  -1  1   5309952  ultralytics

100%|██████████| 5.35M/5.35M [00:00<00:00, 70.4MB/s]


AMP: checks passed ✅
WARNING ⚠️ imgsz=[724] must be multiple of max stride 32, updating to [736]
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 65.6±56.8 MB/s, size: 2780.0 KB)


...

                   all         14         28      0.892      0.885       0.94      0.908
Speed: 0.2ms preprocess, 22.3ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/train
Ultralytics 8.3.159 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                        CUDA:1 (Tesla T4, 15095MiB)
YOLO11x summary (fused): 190 layers, 56,828,179 parameters, 0 gradients, 194.4 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.4 ms, read: 3288.8±627.5 MB/s, size: 4013.2 KB)


val: Scanning /kaggle/input/simulated-object-data/data/val/labels... 14 images, 0 backgrounds, 0 corrupt: 100%|██████████| 14/14 [00:00<00:00, 87.55it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/simulated-object-data/data/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.68it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all         14         28      0.892      0.884       0.94      0.921
Speed: 0.4ms preprocess, 55.7ms inference, 0.0ms loss, 15.9ms postprocess per image
Results saved to runs/detect/train2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ded051b85d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.0

In [10]:
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

In [11]:
output_dir = r"/kaggle/working/predictions/labels"
os.makedirs(output_dir, exist_ok=True)

In [12]:
for i in os.listdir('/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images'):
    img_path = f'/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images/{i}'
    results = model.predict(img_path, 
                            conf=0.3, device=0, verbose=False) # 0 - GPU or "cpu" Image.fromarray(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2HSV))
    output_txt = f"{output_dir}/{i.split('.')[0]}.txt"

    with open(output_txt, "w") as f:
        found = False
        for result in results:
            img_height, img_width = result.orig_shape
            boxes = result.boxes.data

            if boxes is None or len(boxes) == 0:
                continue

            filtered_boxes = boxes[boxes[:, 4] >= 0.05]
            if len(filtered_boxes) == 0:
                continue

            found = True
            for box in filtered_boxes:
                x1, y1, x2, y2, confidence, cls_id = box.tolist()

                x_center = ((x1 + x2) / 2) / img_width
                y_center = ((y1 + y2) / 2) / img_height
                width = (x2 - x1) / img_width
                height = (y2 - y1) / img_height

                f.write(f"0 {confidence:.6f} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

        if not found:
            f.write("")

In [13]:
rows = []
output_dir = Path("/kaggle/working/predictions/labels")
TEST = Path('/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images')
test_imgs = {p.stem for p in TEST.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
predicted = set()

for file in output_dir.glob("*.txt"):
    name = file.stem
    predicted.add(name)

    try:
        lines = [l.strip() for l in open(file) if len(l.strip().split()) == 6]
    except:
        lines = []

    rows.append({"image_id": name, "prediction_string": " ".join(lines) if lines else "no boxes"})

for name in test_imgs - predicted:
    rows.append({"image_id": name, "prediction_string": "no boxes"})

work_dir = '/kaggle/working'

for filename in os.listdir(work_dir):
    file_path = os.path.join(work_dir, filename)

    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print(f'Error file: {file_path}. Cause: {e}')

rows = pandas.DataFrame(rows)
rows.to_csv("submission.csv", index=False)
rows

Error file: /kaggle/working/runs. Cause: name 'shutil' is not defined
Error file: /kaggle/working/predictions. Cause: name 'shutil' is not defined


,image_id,prediction_string
0,IMG_9681,0 0.981792 0.740182 0.637723 0.231614 0.416030...
1,IMG_9613,0 0.981686 0.413613 0.440066 0.106692 0.199114...
2,IMG_9580,0 0.997459 0.404024 0.693686 0.130097 0.109484...
3,IMG_9771,0 0.979924 0.346522 0.590395 0.065138 0.146162...
4,IMG_9734,0 0.993493 0.342402 0.645375 0.202496 0.273470...
...,...,...
174,IMG_9596,0 0.975328 0.214171 0.450549 0.105430 0.179611...
175,IMG_9696,0 0.991550 0.646356 0.613357 0.363404 0.409492...
176,IMG_9678,0 0.989102 0.682595 0.519000 0.217065 0.382436...
177,IMG_9711,0 0.973676 0.394929 0.715653 0.060472 0.128173
